In [ ]:
!pip install langchain
!pip install openai
!pip install PyPDF2
!pip install faiss-cpu
!pip install tiktoken

In [ ]:
!pip install langchain-huggingface sentence-transformers faiss-cpu pypdf

In [ ]:
!pip install langchain-text-splitters

In [ ]:
!pip install langchain_community
from PyPDF2 import PdfReader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [ ]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
pdfreader=PdfReader('/sma unit-1.pdf')

In [ ]:
from IPython.core.interactiveshell import page
from typing_extensions import Concatenate
raw_text=' '
for i,page in enumerate(pdfreader.pages):
  content=page.extract_text()
  if content:
    raw_text+=content

In [ ]:
raw_text

In [ ]:
text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 400,
    chunk_overlap  = 100,
    length_function = len,
)
texts=text_splitter.split_text(raw_text)

In [ ]:
len(texts)

In [ ]:
document_search=FAISS.from_texts(texts,embeddings)

In [ ]:
document_search

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_huggingface import HuggingFacePipeline # Changed to HuggingFacePipeline
from transformers import pipeline # Import pipeline
from google.colab import userdata
from operator import itemgetter

# Initialize LLM with a local Hugging Face pipeline
repo_id="google/flan-t5-base" # Using a smaller model for local execution

# Create a Hugging Face pipeline for text generation
pipe = pipeline(
    "text2text-generation",
    model=repo_id,
    max_new_tokens=50, # Limit output length to prevent very long responses
    temperature=0.5,
    device=0 # Use GPU if available, else CPU (-1)
)

# Wrap the pipeline in LangChain's HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)

# Define the prompt template
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:\n\n{context}\n\nQuestion: {input}"""
)

# Assuming 'document_search' (FAISS vectorstore) is already initialized from previous cells
retriever = document_search.as_retriever()

# Define a function to format retrieved documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construct the RAG chain using LCEL
retrieval_chain = (
    RunnableParallel(
        context=itemgetter("input") | retriever | format_docs,
        input=itemgetter("input")
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
chain = retrieval_chain

In [ ]:
query = "Key Characteristics of web 1.0"
response = chain.invoke({"input": query})
print(response)